# BERT (starter) — Jigsaw toxic comments

**Preprocessing:** `preprocess_for_bert` (tokenizes only the train/val rows used).

**Quick iteration:** Next cell sets `QUICK_ITERATION = True` — small `max_train_samples` / `max_val_samples`, short `MAX_LENGTH`, so tokenizer + training stay fast. Set **`QUICK_ITERATION = False`** for full experiments (slow, memory-heavy).

**Requires:** `pip install torch transformers`

In [4]:
from pathlib import Path
import sys

# Repo root (contains preprocessing/)
ROOT = Path.cwd().resolve()
for c in (ROOT, ROOT.parent):
    if (c / "preprocessing" / "text_preprocessing.py").exists():
        ROOT = c
        break
sys.path.insert(0, str(ROOT))
# notebooks/ for metrics_helpers
sys.path.insert(0, str(ROOT / "notebooks"))

In [5]:
import os
import time

import numpy as np
import torch
from IPython.display import display
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from preprocessing.text_preprocessing import LABEL_COLUMNS, preprocess_for_bert
from metrics_helpers import multilabel_evaluation_report, per_label_confusion_matrices, torch_parameter_count

def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()
print("Device:", DEVICE)
torch.manual_seed(42)

QUICK_ITERATION = True

if QUICK_ITERATION:
    MAX_LENGTH = 64
    BATCH_SIZE = 16
    _train_n, _val_n = 512, 256
else:
    MAX_LENGTH = 128
    BATCH_SIZE = 8
    _train_n, _val_n = None, None

EPOCHS = 1
LR = 2e-5

data = preprocess_for_bert(
    validation_fraction=0.1,
    random_state=42,
    max_length=MAX_LENGTH,
    return_tensors="pt",
    max_train_samples=_train_n,
    max_val_samples=_val_n,
)


def enc_dict(enc):
    keys = [k for k in ("input_ids", "attention_mask", "token_type_ids") if k in enc]
    return {k: enc[k] for k in keys}


train_enc = enc_dict(data.train_encodings)
val_enc = enc_dict(data.val_encodings)
y_train = data.y_train
y_val = data.y_val

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(LABEL_COLUMNS),
    problem_type="multi_label_classification",
).to(DEVICE)


class EncDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item


def collate(batch):
    keys = list(batch[0].keys())
    out = {k: torch.stack([b[k] for b in batch], dim=0) for k in keys if k != "labels"}
    out["labels"] = torch.stack([b["labels"] for b in batch], dim=0)
    return out


train_loader = DataLoader(EncDataset(train_enc, y_train), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(EncDataset(val_enc, y_val), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

opt = torch.optim.AdamW(model.parameters(), lr=LR)
loss_fn = torch.nn.BCEWithLogitsLoss()

print("QUICK_ITERATION:", QUICK_ITERATION, "| train/val rows:", len(y_train), len(y_val), "| max_length:", MAX_LENGTH)

t0 = time.perf_counter()
model.train()
step = 0
for epoch in range(EPOCHS):
    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop("labels")
        opt.zero_grad()
        out = model(**batch)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        opt.step()
        step += 1
train_seconds = time.perf_counter() - t0
print(f"Training wall time: {train_seconds:.1f} s, optimizer steps: {step}")

Device: mps


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


QUICK_ITERATION: True | train/val rows: 512 256 | max_length: 64
Training wall time: 13.8 s, optimizer steps: 32


In [6]:
model.eval()
all_probs = []
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        batch.pop("labels")  # not needed for inference
        logits = model(**batch).logits
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
prob_val = np.concatenate(all_probs, axis=0)
pred_val = (prob_val >= 0.5).astype(np.int64)

per_label_df, summary = multilabel_evaluation_report(y_val, pred_val, prob_val, LABEL_COLUMNS)
print("Micro / macro F1:", summary)
display(per_label_df)
for name, m in per_label_confusion_matrices(y_val, pred_val, LABEL_COLUMNS).items():
    print(name, "\n", m)

print()
print("--- Proposal summary (record for report / comparison) ---")
print(f"  device: {DEVICE}")
print(f"  training_time_s: {train_seconds:.2f}")
print(f"  num_parameters: {torch_parameter_count(model)}")
if summary["f1_macro"] == 0.0 and summary["f1_micro"] == 0.0:
    print(
        "  Note: F1 can be 0 if every predicted label is 0 at threshold 0.5 (common after 1 smoke epoch)."
        " ROC-AUC may still be > 0.5. Raise EPOCHS or set QUICK_ITERATION=False for real training."
    )

Micro / macro F1: {'f1_micro': 0.0, 'f1_macro': 0.0}


,label,precision,recall,f1,roc_auc
0,toxic,0.0,0.0,0.0,0.651464
1,severe_toxic,0.0,0.0,0.0,0.215686
2,obscene,0.0,0.0,0.0,0.685754
3,threat,0.0,0.0,0.0,0.533333
4,insult,0.0,0.0,0.0,0.583956
5,identity_hate,0.0,0.0,0.0,0.421607


toxic 
 [[229   0]
 [ 27   0]]
severe_toxic 
 [[255   0]
 [  1   0]]
obscene 
 [[241   0]
 [ 15   0]]
threat 
 [[255   0]
 [  1   0]]
insult 
 [[241   0]
 [ 15   0]]
identity_hate 
 [[253   0]
 [  3   0]]

--- Proposal summary (record for report / comparison) ---
  device: mps
  training_time_s: 13.84
  num_parameters: 109486854
  Note: F1 can be 0 if every predicted label is 0 at threshold 0.5 (common after 1 smoke epoch). ROC-AUC may still be > 0.5. Raise EPOCHS or set QUICK_ITERATION=False for real training.


## Notes

- Set **`QUICK_ITERATION = False`** in the training cell for larger `max_train_samples` / `max_val_samples`, longer `MAX_LENGTH`, and better scores (needs more RAM/GPU).
- Add **learning rate schedule**, **weight decay**, and **early stopping** for the proposal experiments.
- **DistilBERT** notebook mirrors this with a smaller model for efficiency comparisons.